<a href="https://colab.research.google.com/github/utjimmyx/python/blob/main/Central_Limit_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ----- Settings -----
np.random.seed(123)
n_samples = 1000                 # number of random samples per panel
sample_sizes = [10, 25, 50]      # sample sizes for each panel

# Population: skewed (exponential) to make the CLT visible; swap in any distribution
def draw_population(size):
    return np.random.exponential(scale=2.0, size=size)

# ----- Simulate sample means -----
sample_means = {
    n: [draw_population(n).mean() for _ in range(n_samples)]
    for n in sample_sizes
}

# ----- Build 3-panel figure -----
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"n = {n}" for n in sample_sizes],
    shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=0.04
)

colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for i, n in enumerate(sample_sizes, start=1):
    fig.add_trace(
        go.Histogram(
            x=sample_means[n],
            nbinsx=30,
            marker=dict(color=colors[i-1], line=dict(color="white", width=1)),
            opacity=0.85,
            name=f"n = {n}",
            hovertemplate="Mean: %{x:.2f}<br>Count: %{y}<extra></extra>"
        ),
        row=1, col=i
    )
    fig.update_xaxes(title_text="Sample mean", row=1, col=i)

fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_layout(
    title=f"Sampling Distribution of the Mean ({n_samples:,} samples each)",
    template="plotly_white",
    height=450, width=1000,
    showlegend=False
)

fig.show()

In [2]:
import numpy as np
import plotly.graph_objects as go

# ----- Settings -----
np.random.seed(123)
n_samples = 1000
sample_sizes = [10, 25, 50]
bins = np.linspace(0, 5, 41)                 # fixed bins so bars can animate smoothly
centers = (bins[:-1] + bins[1:]) / 2

# Population: skewed (exponential, mean = 2); swap in any distribution
def draw_population(size):
    return np.random.exponential(scale=2.0, size=size)

# ----- Simulate sample means and bin them -----
counts = {}
for n in sample_sizes:
    means = np.array([draw_population(n).mean() for _ in range(n_samples)])
    counts[n], _ = np.histogram(means, bins=bins)

ymax = max(c.max() for c in counts.values()) * 1.1

# ----- Build animated figure -----
def make_bar(n):
    return go.Bar(
        x=centers, y=counts[n],
        marker=dict(color="#1f77b4", line=dict(color="white", width=1)),
        hovertemplate="Mean: %{x:.2f}<br>Count: %{y}<extra></extra>"
    )

fig = go.Figure(
    data=[make_bar(sample_sizes[0])],
    frames=[go.Frame(data=[make_bar(n)], name=str(n),
                     layout=go.Layout(title_text=f"Sampling distribution of the mean: n = {n}"))
            for n in sample_sizes]
)

fig.update_layout(
    title=f"Sampling distribution of the mean: n = {sample_sizes[0]}",
    template="plotly_white",
    height=500, width=800,
    xaxis=dict(title="Sample mean", range=[0, 5]),
    yaxis=dict(title="Frequency", range=[0, ymax]),
    bargap=0.02,
    updatemenus=[dict(
        type="buttons", direction="left", x=0.0, y=-0.2,
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=1500, redraw=True),
                                  transition=dict(duration=1000, easing="cubic-in-out"),
                                  fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate", transition=dict(duration=0))]),
        ])],
    sliders=[dict(
        active=0, x=0.15, y=-0.15, len=0.8,
        currentvalue=dict(prefix="Sample size n = "),
        steps=[dict(label=str(n), method="animate",
                    args=[[str(n)], dict(mode="immediate",
                                         frame=dict(duration=0, redraw=True),
                                         transition=dict(duration=800))])
               for n in sample_sizes])]
)

fig.show()
fig